# Bài 8 — Đo tốc độ với ViewTransformer

**Mục tiêu:** Chuyển tọa độ pixel → mét thật để tính km/h — **tốc độ hiện ngay trên nhãn từng xe trong cửa sổ live**.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [1]:
!pip install -q supervision ultralytics "supervision[assets]"

In [2]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 16:54:35] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 8.1. Vấn đề phối cảnh

Trên ảnh camera, 100 pixel ở **gần** camera ≠ 100 pixel ở **xa** camera (tính theo mét thật). Giải pháp: **perspective transform** — ánh xạ 4 điểm trên ảnh (hình thang mặt đường) về hình chữ nhật kích thước thật.

```
   Ảnh camera (pixel)                Tọa độ thật (mét)
      A─────B                          A──────B
     ╱       ╲          ──────▶        │      │   vd: rộng 25m
    ╱         ╲                        │      │      dài 250m
   D───────────C                       D──────C
```

## 8.2. ViewTransformer + đo tọa độ SOURCE

 **4 điểm SOURCE và TARGET_WIDTH/HEIGHT dưới đây chỉ là ví dụ minh họa** — dùng tool click chuột ở Bài 6.2 để đo đúng 4 góc mặt đường theo video thật của bạn.

In [5]:
import cv2
import numpy as np
import supervision as sv
from collections import defaultdict, deque
from ultralytics import YOLO
from display import show_frame, close_windows

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "bai8_output.mp4"
VEHICLE_CLASSES = [2, 3, 5, 7]
MAX_FRAMES = 300   # test nhanh trước; đặt None để chạy hết video

# 4 điểm hình thang trên ảnh — PHẢI đo lại theo video thực tế (Bài 6.2)
SOURCE = np.array([[1252, 787], [2298, 803], [5039, 2159], [-550, 2159]])

# Kích thước thật của vùng đó (mét). Mẹo: dựa vào vạch kẻ đường
TARGET_WIDTH, TARGET_HEIGHT = 25, 250
TARGET = np.array([
    [0, 0], [TARGET_WIDTH - 1, 0],
    [TARGET_WIDTH - 1, TARGET_HEIGHT - 1], [0, TARGET_HEIGHT - 1],
])


class ViewTransformer:
    def __init__(self, source: np.ndarray, target: np.ndarray):
        self.m = cv2.getPerspectiveTransform(
            source.astype(np.float32), target.astype(np.float32))

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points
        reshaped = points.reshape(-1, 1, 2).astype(np.float32)
        return cv2.perspectiveTransform(reshaped, self.m).reshape(-1, 2)


view_transformer = ViewTransformer(SOURCE, TARGET)

# Lịch sử tọa độ y (mét) của từng track trong 1 giây gần nhất
model = YOLO("yolov8n.pt")
CLASS_NAMES = model.names   # dict {class_id: ten_class} — dùng thay vì detections.data["class_name"]
# (data["class_name"] bị ByteTrack loại bỏ sau khi update_with_detections, nên không dùng được nữa)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
coordinates = defaultdict(lambda: deque(maxlen=int(video_info.fps)))

tracker = sv.ByteTrack(frame_rate=video_info.fps)
box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.TRACK)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13660\3004145186.py:45: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(frame_rate=video_info.fps)


## 8.3.  Debug bằng mắt: vẽ hình thang SOURCE lên khung hình

Kiểm tra ngay xem 4 điểm chọn có khớp mặt đường không — sai là thấy liền trên cửa sổ.

In [6]:
check_frame = next(sv.get_video_frames_generator(SOURCE_VIDEO))
check_frame = sv.draw_polygon(check_frame.copy(), polygon=SOURCE, color=sv.Color.RED, thickness=4)
show_frame(check_frame, window_name="Kiem tra vung SOURCE", wait=0)
close_windows()

## 8.4. Tính tốc độ km/h theo tracker_id

In [7]:
def compute_speed_labels(detections):
    points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    points = view_transformer.transform_points(points)

    labels = []
    for tracker_id, (_, y) in zip(detections.tracker_id, points):
        coordinates[tracker_id].append(y)
        if len(coordinates[tracker_id]) < video_info.fps / 2:
            labels.append(f"#{tracker_id}")
        else:
            # quãng đường (m) đi được trong khoảng thời gian (s), trung bình 1s để khử jitter
            distance = abs(coordinates[tracker_id][-1] - coordinates[tracker_id][0])
            time_s = len(coordinates[tracker_id]) / video_info.fps
            speed_kmh = distance / time_s * 3.6
            labels.append(f"#{tracker_id} {int(speed_kmh)} km/h")
    return labels


def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, imgsz=640, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, VEHICLE_CLASSES)]
    detections = tracker.update_with_detections(detections)

    labels = compute_speed_labels(detections)

    annotated = frame.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    return annotated

In [8]:
tracker.reset()
coordinates.clear()

sink = sv.VideoSink(target_path=TARGET_VIDEO, video_info=video_info)
sink.__enter__()
try:
    for i, frame in enumerate(sv.get_video_frames_generator(SOURCE_VIDEO)):
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            break
        annotated = process_frame(frame)
        sink.write_frame(annotated)
        if not show_frame(annotated):
            print("Nguoi dung bam Q - dung som.")
            break
finally:
    sink.__exit__(None, None, None)
    close_windows()

print("Xong! Video da luu tai:", TARGET_VIDEO)

Xong! Video da luu tai: bai8_output.mp4


>  **Độ chính xác phụ thuộc:** (1) camera cố định, (2) 4 điểm SOURCE đo đúng, (3) kích thước TARGET đúng. Sai số ±5–10% là bình thường với phương pháp đơn camera.

##  Checkpoint Bài 8

Trên cửa sổ live, nhãn mỗi xe hiện tốc độ km/h, giá trị ổn định (không nhảy loạn).

**Câu hỏi tự kiểm tra:** Vì sao phải dùng `deque` trung bình 1 giây thay vì tính giữa 2 frame liên tiếp?
*(Đáp án: khử nhiễu jitter của box — box detect dao động vài pixel mỗi frame do sai số model, nếu tính tốc độ giữa 2 frame sát nhau thì nhiễu này bị khuếch đại rất lớn khi nhân với 3.6 và fps.)*